# AlterNet 2.0 - Network Inference (MAGNet)

This notebook runs GRNBoost2 inference for three networks using MAGNet cardiac data:
1. **Network 1 (Canonical)**: TF Gene → Target Gene
2. **Network 2 (AS-Aware Source)**: TF Transcript → Target Gene
3. **Network 3 (Fully AS-Aware)**: Regulator Transcript → Target Transcript

## Setup and Imports

In [30]:
import sys
sys.path.append('/home/iwbn121h/alternet2_env/lib/python3.12/site-packages/')

import pandas as pd
import numpy as np
import yaml
import time
import os
import os.path as op

# AlterNet imports
from alternet.data_preprocessing import create_hybrid_data, standardize_dataframe
from alternet.annotation import map_tf_ids, create_transcript_mapping
from alternet.annotation import create_filtered_gene_to_transcripts_mapping
from alternet.annotation import create_transcipt_annotation_database
import alternet.annotation as annotation
import alternet.postprocessing as postprocessing
from alternet.inference import inference

## Configuration

In [31]:
# === PATHS ===
data_path = "./"
results_path = "./results_magnet_net_infer/"

# Reference files (same as GTEx)
appris_path = "appris_data.appris.txt"
digger_path = "digger_data.csv"
biomart_path = "biomart.txt"
tf_list_path = "allTFs_hg38.txt"
sf_list_path = "comprehensive_sfs.csv"

# === MAGNet SPECIFIC ===
# Condition to analyze: 'DCM', 'HCM', or 'NF'
CONDITION = "NF"

# Path to MAGNet pre-filtered TPM file
magnet_tpm_path = f"{CONDITION}_magnet_prefiltered_tpm.tsv"

# Number of GRNBoost2 runs
N_RUNS = 10

# Whether to apply additional variance filtering (optional for MAGNet)
APPLY_VARIANCE_FILTER = True
VARIANCE_PERCENTILE = 0.7  # Keep top 30%

os.makedirs(results_path, exist_ok=True)
print(f"Condition: {CONDITION}")
print(f"Data file: {magnet_tpm_path}")

Condition: NF
Data file: NF_magnet_prefiltered_tpm.tsv
Results path: ./results_magnet_p1/


## Helper Functions

In [32]:
def write_dict_to_yaml(data, filepath):
    """Write dictionary to YAML file."""
    with open(filepath, 'w') as f:
        yaml.dump(data, f, default_flow_style=False)

def map_sf_ids(sf_list_raw, biomart):
    """Map SF gene names to gene and transcript IDs."""
    sf_list_raw = sf_list_raw.copy()
    sf_list_raw.columns = ['SF']
    sf_list = sf_list_raw.merge(biomart, left_on='SF', right_on='Gene name')
    sf_list = sf_list.loc[:, ['SF', 'Gene stable ID', 'Transcript stable ID']].drop_duplicates()
    return sf_list

def combine_tf_sf_lists(tf_list, sf_list):
    """Combine TF and SF lists, marking overlapping genes as TF_SF."""
    tf_list = tf_list.copy()
    sf_list = sf_list.copy()
    
    tf_list['Regulator_type'] = 'TF'
    sf_list['Regulator_type'] = 'SF'
    tf_list = tf_list.rename(columns={'TF': 'Regulator_name'})
    sf_list = sf_list.rename(columns={'SF': 'Regulator_name'})
    
    tf_genes = set(tf_list['Gene stable ID'])
    sf_genes = set(sf_list['Gene stable ID'])
    overlap_genes = tf_genes & sf_genes
    
    print(f"TF genes: {len(tf_genes)}")
    print(f"SF genes: {len(sf_genes)}")
    print(f"Overlap (TF+SF): {len(overlap_genes)}")
    
    combined = pd.concat([tf_list, sf_list], ignore_index=True)
    combined.loc[combined['Gene stable ID'].isin(overlap_genes), 'Regulator_type'] = 'TF_SF'
    combined = combined.drop_duplicates(subset=['Transcript stable ID'], keep='first')
    
    return combined

## Load Reference Data

In [33]:
# Load reference files
biomart = pd.read_csv(biomart_path, sep='\t')
appris_df = pd.read_csv(appris_path, sep='\t')
digger_df = pd.read_csv(digger_path, low_memory=False)

print(f"BioMart entries: {len(biomart)}")
print(f"APPRIS entries: {len(appris_df)}")
print(f"DIGGER entries: {len(digger_df)}")

BioMart entries: 278220
APPRIS entries: 170712
DIGGER entries: 944451


In [34]:
# Load and map TF list
tf_list_raw = pd.read_csv(tf_list_path, sep='\t', header=None)
tf_list = map_tf_ids(tf_list_raw, biomart)

print(f"Unique TF genes: {tf_list['Gene stable ID'].nunique()}")
print(f"Unique TF transcripts: {tf_list['Transcript stable ID'].nunique()}")

Unique TF genes: 1948
Unique TF transcripts: 16298


In [35]:
# Load and map SF list
sf_list_raw = pd.read_csv(sf_list_path, header=None)
sf_list = map_sf_ids(sf_list_raw, biomart)

print(f"Unique SF genes: {sf_list['Gene stable ID'].nunique()}")
print(f"Unique SF transcripts: {sf_list['Transcript stable ID'].nunique()}")

Unique SF genes: 264
Unique SF transcripts: 3227


In [36]:
# Combine TF and SF lists
regulator_list = combine_tf_sf_lists(tf_list, sf_list)

print(f"\nTotal unique genes: {regulator_list['Gene stable ID'].nunique()}")
print(f"Total unique transcripts: {regulator_list['Transcript stable ID'].nunique()}")
print(f"\nBy regulator type:")
print(regulator_list['Regulator_type'].value_counts())

TF genes: 1948
SF genes: 264
Overlap (TF+SF): 42

Total unique genes: 2170
Total unique transcripts: 18958

By regulator type:
Regulator_type
TF       15731
SF        2660
TF_SF      567
Name: count, dtype: int64


In [37]:
# Create mappings
transcript_mapper = annotation.create_transcript_mapping(biomart)
print(f"Transcript-to-gene mappings: {len(transcript_mapper)}")

# Annotation databases
tf_database = annotation.create_transcipt_annotation_database(
    tf_list=tf_list, appris_df=appris_df, digger=digger_df
)
regulator_database = annotation.create_transcipt_annotation_database(
    tf_list=regulator_list, appris_df=appris_df, digger=digger_df
)
print(f"TF annotation database: {len(tf_database)} entries")
print(f"Regulator annotation database: {len(regulator_database)} entries")

Transcript-to-gene mappings: 278220
TF annotation database: 16298 entries
Regulator annotation database: 18958 entries


## Load MAGNet Expression Data

In [38]:
# Load MAGNet transcript TPM data (already pre-filtered by AlterNet 1.0)
transcript_data_raw = pd.read_csv(magnet_tpm_path, sep='\t')

print(f"Loaded MAGNet {CONDITION} data:")
print(f"  Shape: {transcript_data_raw.shape}")
print(f"  Columns: {transcript_data_raw.columns[:5].tolist()} ...")
print(f"\nFirst few rows:")
transcript_data_raw.head()

Loaded MAGNet NF data:
  Shape: (41982, 168)
  Columns: ['transcript_id', 'gene_id', 'SRR10676821', 'SRR10676822', 'SRR10676823'] ...

First few rows:


,transcript_id,gene_id,SRR10676821,SRR10676822,SRR10676823,SRR10676824,SRR10676825,SRR10676826,SRR10676827,SRR10676828,...,SRR10677161,SRR10677163,SRR10677168,SRR10677171,SRR10677174,SRR10677175,SRR10677176,SRR10677177,SRR10677179,SRR10677184
0,ENST00000419783,ENSG00000233276,18.182400,23.665400,58.730700,10.415700,43.194400,55.551300,44.604100,29.813400,...,7.469420,53.473300,87.091700,7.828650,142.587000,86.153100,15.702000,33.564000,149.306000,9.579430
1,ENST00000419349,ENSG00000233276,0.292345,0.181541,0.883829,0.243651,0.604892,0.508725,0.322050,0.000000,...,0.663738,0.207567,0.544754,0.424345,1.000220,0.116289,0.400908,0.554897,0.418136,0.433982
2,ENST00000643797,ENSG00000233276,0.000000,0.000000,0.359728,0.000000,0.649254,0.000000,0.210922,0.000000,...,0.000000,0.612003,0.000000,0.000000,1.275920,1.083740,0.161790,0.000000,1.907470,0.000000
3,ENST00000531391,ENSG00000166473,1.374890,0.889535,0.034239,0.571389,0.294203,0.925668,0.805371,2.142800,...,0.526265,0.390488,1.620090,1.165040,1.114420,0.749224,1.136220,2.119570,0.329100,0.828792
4,ENST00000527937,ENSG00000166473,0.583894,0.140139,0.000000,0.225796,0.023858,0.011475,0.082714,0.591058,...,0.098087,0.215930,0.269296,0.264715,0.016233,0.069791,0.477872,0.198170,0.018419,0.252184


## Preprocess Expression Data

In [39]:
# Standardize column names
transcript_data = transcript_data_raw.copy()

sample_cols = [c for c in transcript_data.columns if c not in ['transcript_id', 'gene_id']]
print(f"Transcripts: {len(transcript_data)}, Samples: {len(sample_cols)}")

Original columns: ['transcript_id', 'gene_id', 'SRR10676821', 'SRR10676822']

Transcripts: 41982, Samples: 166


In [40]:
# Filter to protein-coding transcripts
protein_coding = biomart[biomart['Gene type'] == 'protein_coding']['Transcript stable ID'].unique()
n_before = len(transcript_data)
transcript_data = transcript_data[transcript_data['transcript_id'].isin(protein_coding)].copy()
print(f"Protein-coding filter: {n_before} → {len(transcript_data)} transcripts")

Protein-coding filter: 41982 → 41973 transcripts


In [41]:
# Optional: Variance filter to reduce computational load
if APPLY_VARIANCE_FILTER:
    expression_values = transcript_data[sample_cols].values
    log_expr = np.log1p(expression_values)
    variances = np.var(log_expr, axis=1)
    variance_threshold = np.quantile(variances, VARIANCE_PERCENTILE)
    
    n_before = len(transcript_data)
    transcript_data = transcript_data[variances > variance_threshold].copy()
    print(f"Variance filter (top {int((1-VARIANCE_PERCENTILE)*100)}%): {n_before} → {len(transcript_data)} transcripts")
else:
    print("Variance filter: SKIPPED")

Variance filter (top 30%): 41973 → 12592 transcripts


In [42]:
# Create gene-level data by summing transcript TPMs
gene_data = transcript_data.groupby('gene_id')[sample_cols].sum().reset_index()
print(f"Gene-level data: {len(gene_data)} genes")

Gene-level data: 6661 genes


In [43]:
# Create expression matrices (samples × features)
gene_data_matrix = gene_data.set_index('gene_id')[sample_cols].T
transcript_data_matrix = transcript_data.set_index('transcript_id')[sample_cols].T

print(f"Gene matrix: {gene_data_matrix.shape} (samples × genes)")
print(f"Transcript matrix: {transcript_data_matrix.shape} (samples × transcripts)")

# Standardize (z-score)
gene_data_scaled = standardize_dataframe(gene_data_matrix)
transcript_data_scaled = standardize_dataframe(transcript_data_matrix)

Gene matrix: (166, 6661) (samples × genes)
Transcript matrix: (166, 12592) (samples × transcripts)


In [44]:
# Remove problematic transcripts (NaN or zero variance)
nan_cols = transcript_data_scaled.columns[transcript_data_scaled.isna().any()].tolist()
zero_var_cols = transcript_data_matrix.columns[transcript_data_matrix.std() == 0].tolist()
bad_transcripts = set(nan_cols + zero_var_cols)

if bad_transcripts:
    good_transcripts = [c for c in transcript_data_scaled.columns if c not in bad_transcripts]
    transcript_data_scaled = transcript_data_scaled[good_transcripts]
    transcript_data_matrix = transcript_data_matrix[good_transcripts]
    print(f"Removed {len(bad_transcripts)} problematic transcripts")

# Fill remaining NaN
transcript_data_scaled = transcript_data_scaled.fillna(0)
gene_data_scaled = gene_data_scaled.fillna(0)

print(f"Final: {len(transcript_data_scaled.columns)} transcripts, {len(gene_data_scaled.columns)} genes")

Final: 12592 transcripts, 6661 genes


## Identify Regulators in Data

In [45]:
# Gene-to-transcript mapping for genes in data
gene_to_transcript_mapping = annotation.create_filtered_gene_to_transcripts_mapping(
    biomart,
    gene_list=gene_data_scaled.columns,
    transcript_list=transcript_data_scaled.columns
)
print(f"Gene-to-transcript mappings: {len(gene_to_transcript_mapping)}")

Gene-to-transcript mappings: 6661


In [46]:
# TF genes in data (for Network 1)
tf_genes_in_data = list(set(tf_list['Gene stable ID']) & set(gene_data_scaled.columns))
print(f"TF genes in data: {len(tf_genes_in_data)}")

# TF transcripts in data (for Network 2)
tf_transcripts_in_data = list(set(tf_list['Transcript stable ID']) & set(transcript_data_scaled.columns))
print(f"TF transcripts in data: {len(tf_transcripts_in_data)}")

# All regulator transcripts (for Network 3)
regulator_transcripts_in_data = list(
    set(regulator_list['Transcript stable ID']) & set(transcript_data_scaled.columns)
)
print(f"All regulator transcripts (TF+SF) in data: {len(regulator_transcripts_in_data)}")

# Targets
target_genes = list(gene_data_scaled.columns)
target_transcripts = list(transcript_data_scaled.columns)
print(f"Target genes: {len(target_genes)}")
print(f"Target transcripts: {len(target_transcripts)}")

TF genes in data: 553
TF transcripts in data: 1054
All regulator transcripts (TF+SF) in data: 1298
Target genes: 6661
Target transcripts: 12592


In [47]:
# Regulator types in data
tf_only_transcripts = set(regulator_list[regulator_list['Regulator_type'] == 'TF']['Transcript stable ID'])
sf_only_transcripts = set(regulator_list[regulator_list['Regulator_type'] == 'SF']['Transcript stable ID'])
tfsf_transcripts = set(regulator_list[regulator_list['Regulator_type'] == 'TF_SF']['Transcript stable ID'])

tf_only_transcripts = tf_only_transcripts & set(transcript_data_scaled.columns)
sf_only_transcripts = sf_only_transcripts & set(transcript_data_scaled.columns)
tfsf_transcripts = tfsf_transcripts & set(transcript_data_scaled.columns)

print(f"TF only: {len(tf_only_transcripts)}")
print(f"SF only: {len(sf_only_transcripts)}")
print(f"TF+SF: {len(tfsf_transcripts)}")

TF only: 995
SF only: 244
TF+SF: 59


## Compute Isoform Categories

In [48]:
# TF isoform categories (for Network 1 & 2 comparison)
tf_isoform_categories = postprocessing.isoform_categorization(
    transcript_data_matrix, gene_data_matrix, tf_list
)
tf_gene_categories = postprocessing.get_gene_cases(tf_isoform_categories)

print("TF isoform categories:")
print(tf_isoform_categories['isoform_category'].value_counts())

TF isoform categories:
isoform_category
non-dominant    463
single          293
balanced        244
dominant         54
Name: count, dtype: int64


In [49]:
# Full regulator isoform categories (TF + SF, for Network 3)
regulator_isoform_categories = postprocessing.isoform_categorization(
    transcript_data_matrix, gene_data_matrix, regulator_list
)
regulator_gene_categories = postprocessing.get_gene_cases(regulator_isoform_categories)

print("Regulator isoform categories:")
print(regulator_isoform_categories['isoform_category'].value_counts())

Regulator isoform categories:
isoform_category
non-dominant    578
single          349
balanced        297
dominant         74
Name: count, dtype: int64


In [50]:
# Target isoform categories (for Network 3)
all_genes_as_targets = biomart[['Gene stable ID', 'Transcript stable ID']].drop_duplicates()
all_genes_as_targets = all_genes_as_targets[all_genes_as_targets['Gene stable ID'].isin(gene_data_scaled.columns)]
all_genes_as_targets = all_genes_as_targets[all_genes_as_targets['Transcript stable ID'].isin(transcript_data_scaled.columns)]

target_isoform_categories = postprocessing.isoform_categorization(
    transcript_data_matrix, gene_data_matrix, all_genes_as_targets
)
target_gene_categories = postprocessing.get_gene_cases(target_isoform_categories)

print("Target isoform categories:")
print(target_isoform_categories['isoform_category'].value_counts())

Target isoform categories:
isoform_category
non-dominant    5956
single          3481
balanced        2371
dominant         784
Name: count, dtype: int64


## Run Network Inference

In [51]:
runtime = {}

In [52]:
# NETWORK 1: Canonical
print("NETWORK 1: Canonical")
print(f"Regulators: {len(tf_genes_in_data)} TF genes")
print(f"Targets: {len(target_genes)} genes")

start = time.monotonic()
canonical_grn = inference(
    gene_data=gene_data_scaled,
    tf_list=tf_genes_in_data,
    target_names='all',
    n_runs=N_RUNS
)
runtime['canonical'] = time.monotonic() - start

print(f"\nEdges: {len(canonical_grn):,}")
print(f"Time: {runtime['canonical']/60:.2f} minutes")

# Save
canonical_grn.to_csv(op.join(results_path, f"{CONDITION}_canonical_raw.tsv"), sep='\t', index=False)

NETWORK 1: Canonical
Regulators: 553 TF genes
Targets: 6661 genes


/home/hpc/iwbn/iwbn121h/alternet2_env/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 46497 instead
  warnings.warn(
  0%|                                                    | 0/10 [00:00<?, ?it/s]/home/hpc/iwbn/iwbn121h/alternet2_env/lib/python3.12/site-packages/distributed/client.py:3374: UserWarning: Sending large graph of size 11.23 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
 10%|████▎                                      | 1/10 [04:45<42:46, 285.19s/it]/home/hpc/iwbn/iwbn121h/alternet2_env/lib/python3.12/site-packages/distributed/client.py:3374: UserWarning: Sending large graph of size 11.23 MiB.
This may cause some 


Edges: 2,563,477
Time: 45.91 minutes
Saved: NF_canonical_raw.tsv


In [53]:
# NETWORK 2: AS-Aware Source
print("NETWORK 2: AS-Aware Source")
print(f"Regulators: {len(tf_transcripts_in_data)} TF transcripts")
print(f"Targets: {len(target_genes)} genes")

# Create hybrid data (TF transcripts + target genes)
hybrid_data = create_hybrid_data(
    transcript_data_matrix,  
    gene_data_matrix,        
    tf_list
)
print(f"Hybrid data shape: {hybrid_data.shape}")

start = time.monotonic()
as_source_grn = inference(
    gene_data=hybrid_data,
    tf_list=tf_transcripts_in_data,
    target_names=target_genes,
    n_runs=N_RUNS
)
runtime['as_aware_source'] = time.monotonic() - start

print(f"\nEdges: {len(as_source_grn):,}")
print(f"Time: {runtime['as_aware_source']/60:.2f} minutes")

# Save
as_source_grn.to_csv(op.join(results_path, f"{CONDITION}_as_aware_source_raw.tsv"), sep='\t', index=False)

NETWORK 2: AS-Aware Source
Regulators: 1054 TF transcripts
Targets: 6661 genes
Hybrid data shape: (166, 7715)


/home/hpc/iwbn/iwbn121h/alternet2_env/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 38539 instead
  warnings.warn(
  0%|                                                    | 0/10 [00:00<?, ?it/s]/home/hpc/iwbn/iwbn121h/alternet2_env/lib/python3.12/site-packages/distributed/client.py:3374: UserWarning: Sending large graph of size 11.23 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
 10%|████▎                                      | 1/10 [06:00<54:07, 360.85s/it]/home/hpc/iwbn/iwbn121h/alternet2_env/lib/python3.12/site-packages/distributed/client.py:3374: UserWarning: Sending large graph of size 11.23 MiB.
This may cause some 


Edges: 3,521,313
Time: 57.90 minutes
Saved: NF_as_aware_source_raw.tsv


In [54]:
# NETWORK 3: Fully AS-Aware
print("NETWORK 3: Fully AS-Aware")
print(f"Regulators: {len(regulator_transcripts_in_data)} transcripts (TF+SF)")
print(f"Targets: {len(target_transcripts)} transcripts")

start = time.monotonic()
fully_as_grn = inference(
    gene_data=transcript_data_scaled,
    tf_list=regulator_transcripts_in_data,
    target_names='all',
    n_runs=N_RUNS
)
runtime['fully_as_aware'] = time.monotonic() - start

print(f"\nEdges: {len(fully_as_grn):,}")
print(f"Time: {runtime['fully_as_aware']/60:.2f} minutes")

# Save
fully_as_grn.to_csv(op.join(results_path, f"{CONDITION}_fully_as_aware_raw.tsv"), sep='\t', index=False)

NETWORK 3: Fully AS-Aware
Regulators: 1298 transcripts (TF+SF)
Targets: 12592 transcripts


/home/hpc/iwbn/iwbn121h/alternet2_env/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41003 instead
  warnings.warn(
  0%|                                                    | 0/10 [00:00<?, ?it/s]/home/hpc/iwbn/iwbn121h/alternet2_env/lib/python3.12/site-packages/distributed/client.py:3374: UserWarning: Sending large graph of size 21.23 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
 10%|████                                     | 1/10 [11:26<1:42:58, 686.47s/it]/home/hpc/iwbn/iwbn121h/alternet2_env/lib/python3.12/site-packages/distributed/client.py:3374: UserWarning: Sending large graph of size 21.23 MiB.
This may cause some 


Edges: 7,517,491
Time: 114.02 minutes
Saved: NF_fully_as_aware_raw.tsv


In [55]:
# Save runtime
runtime['total'] = runtime['canonical'] + runtime['as_aware_source'] + runtime['fully_as_aware']
write_dict_to_yaml(runtime, op.join(results_path, f"{CONDITION}_runtime.yaml"))
print(f"\nTotal inference time: {runtime['total']/60:.2f} minutes")


Total inference time: 217.84 minutes


## Save Metadata

In [56]:
# Save isoform categories
tf_isoform_categories.to_csv(
    op.join(results_path, f"{CONDITION}_tf_isoform_categories.csv"), index=False
)
regulator_isoform_categories.to_csv(
    op.join(results_path, f"{CONDITION}_regulator_isoform_categories.csv"), index=False
)
target_isoform_categories.to_csv(
    op.join(results_path, f"{CONDITION}_target_isoform_categories.csv"), index=False
)

# Save gene categories
tf_gene_categories.to_csv(
    op.join(results_path, f"{CONDITION}_tf_gene_categories.csv"), index=False
)
regulator_gene_categories.to_csv(
    op.join(results_path, f"{CONDITION}_regulator_gene_categories.csv"), index=False
)
target_gene_categories.to_csv(
    op.join(results_path, f"{CONDITION}_target_gene_categories.csv"), index=False
)

# Save regulator list with types
regulator_list.to_csv(
    op.join(results_path, f"{CONDITION}_regulator_list.csv"), index=False
)


Saved all metadata files


In [57]:
# Save summary statistics
summary_stats = {
    'dataset': 'MAGNet',
    'condition': CONDITION,
    'n_samples': len(sample_cols),
    'n_genes': len(gene_data_scaled.columns),
    'n_transcripts': len(transcript_data_scaled.columns),
    'n_tf_genes': len(tf_genes_in_data),
    'n_tf_transcripts': len(tf_transcripts_in_data),
    'n_sf_transcripts': len(sf_only_transcripts),
    'n_tfsf_transcripts': len(tfsf_transcripts),
    'n_regulator_transcripts': len(regulator_transcripts_in_data),
    'network1_edges': len(canonical_grn),
    'network2_edges': len(as_source_grn),
    'network3_edges': len(fully_as_grn),
    'runtime_canonical_min': runtime['canonical'] / 60,
    'runtime_as_source_min': runtime['as_aware_source'] / 60,
    'runtime_fully_as_min': runtime['fully_as_aware'] / 60,
    'runtime_total_min': runtime['total'] / 60,
}

write_dict_to_yaml(summary_stats, op.join(results_path, f"{CONDITION}_summary_stats.yaml"))

Saved summary statistics
